# Домашняя работа по C#

Ноутбук содержит функции для рисования фигур в консольном стиле.

Реализовано:
- `GetRectangle(...)`, `PrintRectangle(...)`
- `GetRightIsoscelesTriangle(...)`
- `GetHourglass(...)`
- `GetRhombus(...)`
- `GetZebra(...)`
- `GetSnake(...)` — горизонтальная и вертикальная змейка
- `GetNestedRectangles(...)`
- `GetSpiral(...)` — улитка по часовой и против часовой

Во всех случаях фигуры возвращаются в виде массива строк (`string[]`), а отдельные `Print...`-функции выводят результат на экран.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public enum Rotation
{
    Clockwise,
    CounterClockwise
}

public enum Orientation
{
    Horizontal,
    Vertical
}

public static class ShapePrinter
{
    private static string[] CanvasToLines(char[,] canvas)
    {
        int rows = canvas.GetLength(0);
        int cols = canvas.GetLength(1);
        var result = new string[rows];

        for (int r = 0; r < rows; r++)
        {
            char[] line = new char[cols];
            for (int c = 0; c < cols; c++)
                line[c] = canvas[r, c];

            result[r] = new string(line);
        }

        return result;
    }

    private static void ValidatePositive(int value, string name)
    {
        if (value <= 0)
            throw new ArgumentException($"{name} должно быть больше 0");
    }

    private static char[,] CreateCanvas(int height, int width, char background = ' ')
    {
        var canvas = new char[height, width];
        for (int r = 0; r < height; r++)
            for (int c = 0; c < width; c++)
                canvas[r, c] = background;
        return canvas;
    }

    public static void PrintShape(string[] shape)
    {
        foreach (var line in shape)
            Console.WriteLine(line);
    }

    public static string[] GetRectangle(
        int width,
        int height,
        char borderChar = '*',
        bool filled = false,
        char fillChar = ' ',
        int borderThickness = 1)
    {
        ValidatePositive(width, nameof(width));
        ValidatePositive(height, nameof(height));
        ValidatePositive(borderThickness, nameof(borderThickness));

        if (borderThickness * 2 > width || borderThickness * 2 > height)
            throw new ArgumentException("Толщина рамки слишком большая для заданных размеров");

        var canvas = CreateCanvas(height, width, ' ');

        for (int r = 0; r < height; r++)
        {
            for (int c = 0; c < width; c++)
            {
                bool isBorder =
                    r < borderThickness ||
                    r >= height - borderThickness ||
                    c < borderThickness ||
                    c >= width - borderThickness;

                if (isBorder)
                    canvas[r, c] = borderChar;
                else if (filled)
                    canvas[r, c] = fillChar;
            }
        }

        return CanvasToLines(canvas);
    }

    public static void PrintRectangle(
        int width,
        int height,
        char borderChar = '*',
        bool filled = false,
        char fillChar = ' ',
        int borderThickness = 1)
    {
        PrintShape(GetRectangle(width, height, borderChar, filled, fillChar, borderThickness));
    }

    public static string[] GetRightIsoscelesTriangle(
        int leg,
        char borderChar = '*',
        bool filled = false,
        char fillChar = ' ')
    {
        ValidatePositive(leg, nameof(leg));

        var lines = new List<string>();

        for (int r = 0; r < leg; r++)
        {
            char[] row = Enumerable.Repeat(' ', leg).ToArray();

            for (int c = 0; c <= r; c++)
            {
                bool isBorder = r == leg - 1 || c == 0 || c == r;
                if (isBorder)
                    row[c] = borderChar;
                else if (filled)
                    row[c] = fillChar;
            }

            lines.Add(new string(row));
        }

        return lines.ToArray();
    }

    public static void PrintRightIsoscelesTriangle(
        int leg,
        char borderChar = '*',
        bool filled = false,
        char fillChar = ' ')
    {
        PrintShape(GetRightIsoscelesTriangle(leg, borderChar, filled, fillChar));
    }

    public static string[] GetHourglass(
        int halfHeight,
        char drawChar = '*',
        bool filled = false,
        char fillChar = ' ')
    {
        ValidatePositive(halfHeight, nameof(halfHeight));
        int size = halfHeight * 2 - 1;
        var lines = new List<string>();

        for (int r = 0; r < size; r++)
        {
            int offset = r < halfHeight ? r : size - 1 - r;
            int innerWidth = size - 2 * offset;
            char[] row = Enumerable.Repeat(' ', size).ToArray();

            for (int i = 0; i < innerWidth; i++)
            {
                int c = offset + i;
                bool isBorder = i == 0 || i == innerWidth - 1 || r == 0 || r == size - 1;
                if (innerWidth == 1 || isBorder)
                    row[c] = drawChar;
                else if (filled)
                    row[c] = fillChar;
            }

            lines.Add(new string(row));
        }

        return lines.ToArray();
    }

    public static void PrintHourglass(
        int halfHeight,
        char drawChar = '*',
        bool filled = false,
        char fillChar = ' ')
    {
        PrintShape(GetHourglass(halfHeight, drawChar, filled, fillChar));
    }

    public static string[] GetRhombus(
        int radius,
        char borderChar = '*',
        bool filled = false,
        char fillChar = ' ')
    {
        ValidatePositive(radius, nameof(radius));

        int size = radius * 2 - 1;
        var lines = new List<string>();

        for (int r = 0; r < size; r++)
        {
            int dist = Math.Abs(radius - 1 - r);
            int inner = size - 2 * dist;
            int start = dist;
            char[] row = Enumerable.Repeat(' ', size).ToArray();

            for (int i = 0; i < inner; i++)
            {
                int c = start + i;
                bool isBorder = i == 0 || i == inner - 1;
                if (inner == 1 || isBorder)
                    row[c] = borderChar;
                else if (filled)
                    row[c] = fillChar;
            }

            lines.Add(new string(row));
        }

        return lines.ToArray();
    }

    public static void PrintRhombus(
        int radius,
        char borderChar = '*',
        bool filled = false,
        char fillChar = ' ')
    {
        PrintShape(GetRhombus(radius, borderChar, filled, fillChar));
    }

    public static string[] GetZebra(
        int width,
        int height,
        int stripeThickness = 1,
        char stripeChar = '#',
        char backgroundChar = ' ')
    {
        ValidatePositive(width, nameof(width));
        ValidatePositive(height, nameof(height));
        ValidatePositive(stripeThickness, nameof(stripeThickness));

        var lines = new List<string>();

        for (int r = 0; r < height; r++)
        {
            bool stripe = (r / stripeThickness) % 2 == 0;
            lines.Add(new string(stripe ? stripeChar : backgroundChar, width));
        }

        return lines.ToArray();
    }

    public static void PrintZebra(
        int width,
        int height,
        int stripeThickness = 1,
        char stripeChar = '#',
        char backgroundChar = ' ')
    {
        PrintShape(GetZebra(width, height, stripeThickness, stripeChar, backgroundChar));
    }

    public static string[] GetSnake(
        int rows,
        int cols,
        Orientation orientation = Orientation.Horizontal,
        Rotation rotation = Rotation.Clockwise,
        char bodyChar = '*',
        char backgroundChar = ' ')
    {
        ValidatePositive(rows, nameof(rows));
        ValidatePositive(cols, nameof(cols));

        var canvas = CreateCanvas(rows, cols, backgroundChar);

        if (orientation == Orientation.Horizontal)
        {
            for (int r = 0; r < rows; r++)
            {
                bool leftToRight =
                    rotation == Rotation.Clockwise
                    ? r % 2 == 0
                    : r % 2 == 1;

                if (leftToRight)
                {
                    for (int c = 0; c < cols; c++)
                        canvas[r, c] = bodyChar;
                }
                else
                {
                    for (int c = cols - 1; c >= 0; c--)
                        canvas[r, c] = bodyChar;
                }
            }
        }
        else
        {
            for (int c = 0; c < cols; c++)
            {
                bool topToBottom =
                    rotation == Rotation.Clockwise
                    ? c % 2 == 0
                    : c % 2 == 1;

                if (topToBottom)
                {
                    for (int r = 0; r < rows; r++)
                        canvas[r, c] = bodyChar;
                }
                else
                {
                    for (int r = rows - 1; r >= 0; r--)
                        canvas[r, c] = bodyChar;
                }
            }
        }

        // Чтобы рисунок выглядел именно как "змейка", добавим пробелы в точках разрыва.
        if (orientation == Orientation.Horizontal)
        {
            for (int r = 1; r < rows; r++)
            {
                bool prevLeftToRight =
                    rotation == Rotation.Clockwise
                    ? (r - 1) % 2 == 0
                    : (r - 1) % 2 == 1;

                if (prevLeftToRight)
                    for (int c = 0; c < cols - 1; c++) canvas[r, c] = backgroundChar;
                else
                    for (int c = 1; c < cols; c++) canvas[r, c] = backgroundChar;
            }
        }
        else
        {
            for (int c = 1; c < cols; c++)
            {
                bool prevTopToBottom =
                    rotation == Rotation.Clockwise
                    ? (c - 1) % 2 == 0
                    : (c - 1) % 2 == 1;

                if (prevTopToBottom)
                    for (int r = 0; r < rows - 1; r++) canvas[r, c] = backgroundChar;
                else
                    for (int r = 1; r < rows; r++) canvas[r, c] = backgroundChar;
            }
        }

        return CanvasToLines(canvas);
    }

    public static void PrintSnake(
        int rows,
        int cols,
        Orientation orientation = Orientation.Horizontal,
        Rotation rotation = Rotation.Clockwise,
        char bodyChar = '*',
        char backgroundChar = ' ')
    {
        PrintShape(GetSnake(rows, cols, orientation, rotation, bodyChar, backgroundChar));
    }

    public static string[] GetNestedRectangles(
        int width,
        int height,
        int layers,
        char drawChar = '*',
        char backgroundChar = ' ')
    {
        ValidatePositive(width, nameof(width));
        ValidatePositive(height, nameof(height));
        ValidatePositive(layers, nameof(layers));

        var canvas = CreateCanvas(height, width, backgroundChar);

        for (int layer = 0; layer < layers; layer++)
        {
            int top = layer * 2;
            int left = layer * 2;
            int bottom = height - 1 - layer * 2;
            int right = width - 1 - layer * 2;

            if (top > bottom || left > right)
                break;

            for (int c = left; c <= right; c++)
            {
                canvas[top, c] = drawChar;
                canvas[bottom, c] = drawChar;
            }

            for (int r = top; r <= bottom; r++)
            {
                canvas[r, left] = drawChar;
                canvas[r, right] = drawChar;
            }
        }

        return CanvasToLines(canvas);
    }

    public static void PrintNestedRectangles(
        int width,
        int height,
        int layers,
        char drawChar = '*',
        char backgroundChar = ' ')
    {
        PrintShape(GetNestedRectangles(width, height, layers, drawChar, backgroundChar));
    }

    public static string[] GetSpiral(
        int width,
        int height,
        Rotation rotation = Rotation.Clockwise,
        char drawChar = '*',
        char backgroundChar = ' ')
    {
        ValidatePositive(width, nameof(width));
        ValidatePositive(height, nameof(height));

        var canvas = CreateCanvas(height, width, backgroundChar);

        int top = 0, left = 0, bottom = height - 1, right = width - 1;

        while (top <= bottom && left <= right)
        {
            if (rotation == Rotation.Clockwise)
            {
                for (int c = left; c <= right; c++) canvas[top, c] = drawChar;
                top++;

                for (int r = top; r <= bottom; r++) canvas[r, right] = drawChar;
                right--;

                if (top <= bottom)
                {
                    for (int c = right; c >= left; c--) canvas[bottom, c] = drawChar;
                    bottom--;
                }

                if (left <= right)
                {
                    for (int r = bottom; r >= top + 1; r--) canvas[r, left] = drawChar;
                    left += 2;
                }
            }
            else
            {
                for (int r = top; r <= bottom; r++) canvas[r, left] = drawChar;
                left++;

                for (int c = left; c <= right; c++) canvas[bottom, c] = drawChar;
                bottom--;

                if (left <= right)
                {
                    for (int r = bottom; r >= top; r--) canvas[r, right] = drawChar;
                    right--;
                }

                if (top <= bottom)
                {
                    for (int c = right; c >= left + 1; c--) canvas[top, c] = drawChar;
                    top += 2;
                }
            }
        }

        return CanvasToLines(canvas);
    }

    public static void PrintSpiral(
        int width,
        int height,
        Rotation rotation = Rotation.Clockwise,
        char drawChar = '*',
        char backgroundChar = ' ')
    {
        PrintShape(GetSpiral(width, height, rotation, drawChar, backgroundChar));
    }
}

## 1. Проверка прямоугольников

Ниже тестируются варианты:
- пустой прямоугольник;
- залитый прямоугольник;
- рамка большей толщины.

In [ ]:
Console.WriteLine("Прямоугольник 8x4, пустой:");
ShapePrinter.PrintRectangle(8, 4);

Console.WriteLine("\nПрямоугольник 10x5, залитый:");
ShapePrinter.PrintRectangle(10, 5, '#', true, '.');

Console.WriteLine("\nПрямоугольник 12x6, рамка толщиной 2:");
ShapePrinter.PrintRectangle(12, 6, '@', false, ' ', 2);

## 2. Проверка равнобедренного прямоугольного треугольника

In [ ]:
Console.WriteLine("Треугольник, leg = 6, без заливки:");
ShapePrinter.PrintRightIsoscelesTriangle(6);

Console.WriteLine("\nТреугольник, leg = 7, с заливкой:");
ShapePrinter.PrintRightIsoscelesTriangle(7, '#', true, '.');

## 3. Проверка песочных часов

In [ ]:
Console.WriteLine("Песочные часы, halfHeight = 5, без заливки:");
ShapePrinter.PrintHourglass(5);

Console.WriteLine("\nПесочные часы, halfHeight = 4, с заливкой:");
ShapePrinter.PrintHourglass(4, '#', true, '.');

## 4. Проверка ромба

In [ ]:
Console.WriteLine("Ромб, radius = 5, без заливки:");
ShapePrinter.PrintRhombus(5);

Console.WriteLine("\nРомб, radius = 4, с заливкой:");
ShapePrinter.PrintRhombus(4, '#', true, '.');

## 5. Проверка зебры

In [ ]:
Console.WriteLine("Зебра 14x8, толщина полосы 1:");
ShapePrinter.PrintZebra(14, 8, 1, '#', '.');

Console.WriteLine("\nЗебра 18x10, толщина полосы 2:");
ShapePrinter.PrintZebra(18, 10, 2, '=', ' ');

## 6. Проверка змейки

Проверяются:
- горизонтальная по часовой;
- горизонтальная против часовой;
- вертикальная по часовой;
- вертикальная против часовой.

In [ ]:
Console.WriteLine("Горизонтальная змейка, clockwise:");
ShapePrinter.PrintSnake(6, 12, Orientation.Horizontal, Rotation.Clockwise, '*', '.');

Console.WriteLine("\nГоризонтальная змейка, counterclockwise:");
ShapePrinter.PrintSnake(6, 12, Orientation.Horizontal, Rotation.CounterClockwise, '*', '.');

Console.WriteLine("\nВертикальная змейка, clockwise:");
ShapePrinter.PrintSnake(8, 7, Orientation.Vertical, Rotation.Clockwise, '*', '.');

Console.WriteLine("\nВертикальная змейка, counterclockwise:");
ShapePrinter.PrintSnake(8, 7, Orientation.Vertical, Rotation.CounterClockwise, '*', '.');

## 7. Проверка вложенных прямоугольников

In [ ]:
Console.WriteLine("Вложенные прямоугольники 21x11, 3 слоя:");
ShapePrinter.PrintNestedRectangles(21, 11, 3, '*', '.');

Console.WriteLine("\nВложенные прямоугольники 17x9, 4 слоя:");
ShapePrinter.PrintNestedRectangles(17, 9, 4, '#', ' ');

## 8. Проверка улитки

Проверяются варианты:
- по часовой стрелке;
- против часовой стрелки.

In [ ]:
Console.WriteLine("Улитка 15x9, clockwise:");
ShapePrinter.PrintSpiral(15, 9, Rotation.Clockwise, '*', '.');

Console.WriteLine("\nУлитка 15x9, counterclockwise:");
ShapePrinter.PrintSpiral(15, 9, Rotation.CounterClockwise, '#', '.');

## 9. Дополнительные тестовые случаи

Здесь показано, что функции можно использовать и как генераторы массива строк, а не только для печати.

In [ ]:
string[] rect = ShapePrinter.GetRectangle(7, 3, '+', true, '-');
Console.WriteLine("Получили массив строк для прямоугольника:");
Console.WriteLine($"Количество строк: {rect.Length}");
Console.WriteLine("Содержимое:");
foreach (var line in rect)
    Console.WriteLine($"[{line}]");

Console.WriteLine();

string[] diamond = ShapePrinter.GetRhombus(3, '%', true, '.');
Console.WriteLine("Получили массив строк для ромба:");
foreach (var line in diamond)
    Console.WriteLine($"[{line}]");